# 01 - Bronze Ingestion: ADLS CSV sources

Reads the 6 operational CSV sources (calendar, promotions, regions, stores,
returns, transactions) that land in Azure Data Lake Storage, and loads them
into Bronze Delta tables with Auto Loader.

Runs as a task inside `medallion_pipeline_job` (see `resources/jobs.yml`).
Catalog name comes in through a widget so the same notebook works for the
`maven_test` and `maven_dev` catalogs without any code change.

In [0]:
# Databricks notebook source
dbutils.widgets.text("catalog", "maven_test")
catalog = dbutils.widgets.get("catalog")

storage_account = "capstoneadls"
landing_container = "landing"

print(f"Running Bronze ADLS ingestion for catalog: {catalog}")

### Make sure the checkpoint volume exists

Auto Loader needs somewhere to keep its checkpoint + schema state between
runs. One volume, one subfolder per table.

In [0]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.bronze.checkpoints")

### Table list

One entry per CSV source: table name -> folder under `landing/`.

In [0]:
import datetime
from pyspark.sql import functions as F

tables = {
    "calendar": "operational/calendar/",
    "promotions": "operational/promotions/",
    "regions": "operational/regions/",
    "stores": "operational/stores/",
    "returns": "pos/returns/",
    "transactions": "pos/transactions/",
}

run_batch_id = "BATCH_" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S")
print("Batch id:", run_batch_id)

### Ingest each table

Auto Loader (`cloudFiles`) picks up new/changed files automatically and
evolves the schema if a new column shows up. We retry each table up to 3
times before giving up on it — one bad table should not stop the other
five from loading.

In [0]:
results = {}

for table_name, folder in tables.items():
    full_table = f"{catalog}.bronze.{table_name}"
    source_path = f"abfss://{landing_container}@{storage_account}.dfs.core.windows.net/{folder}"
    checkpoint_path = f"/Volumes/{catalog}/bronze/checkpoints/{table_name}/state"
    schema_path = f"/Volumes/{catalog}/bronze/checkpoints/{table_name}/schema"

    print(f"\nLoading {full_table} from {source_path}")

    attempt = 0
    done = False
    while attempt < 3 and not done:
        attempt += 1
        try:
            df = (
                spark.readStream
                .format("cloudFiles")
                .option("cloudFiles.format", "csv")
                .option("cloudFiles.schemaLocation", schema_path)
                .option("cloudFiles.inferColumnTypes", "true")
                .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
                .option("cloudFiles.rescuedDataColumn", "_rescued_data")
                .option("header", "true")
                .load(source_path)
                .withColumn("ingestion_timestamp", F.current_timestamp())
                .withColumn("source_file", F.col("_metadata.file_name"))
                .withColumn("source_system", F.lit("ADLS_OPERATIONAL_INGRESS"))
                .withColumn("load_batch_id", F.lit(run_batch_id))
            )

            query = (
                df.writeStream
                .format("delta")
                .option("checkpointLocation", checkpoint_path)
                .option("mergeSchema", "true")
                .trigger(availableNow=True)
                .toTable(full_table)
            )
            query.awaitTermination()

            row_count = spark.table(full_table).count()
            print(f"  ok, {row_count} rows in {full_table}")
            results[table_name] = {"status": "SUCCESS", "row_count": row_count}
            done = True

        except Exception as e:
            print(f"  attempt {attempt} failed for {table_name}: {e}")
            if attempt == 3:
                results[table_name] = {"status": "FAILED", "error": str(e)}

print("\nSummary:", results)

### Fail the task if any table failed

`bronze_validation` (the next task) expects every table to be there — if a
table failed here, raise so the job stops instead of quietly moving on to
Silver/Gold with partial data.

In [0]:
failed = [t for t, r in results.items() if r["status"] == "FAILED"]
if failed:
    raise RuntimeError(f"ADLS ingestion failed for: {failed}")

dbutils.notebook.exit("OK")